SELEZIONE DEI MODELLI MIGLIORI

In [5]:
!pip install m2cgen

In [6]:
!pip install mpy-cross

In [7]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
from sklearn.metrics import mean_absolute_percentage_error
import sys
import os
import ast
from pathlib import Path
import m2cgen as m2c
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
import lightgbm as lgb
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import ElasticNet
from sklearn.preprocessing import StandardScaler
import subprocess
from sklearn.decomposition import PCA

In [8]:
from google.colab import drive
# 1. Collega Drive
drive.mount('/content/drive')

# 2. Vai nella cartella dove hai i file e la cartella audio
%cd "/content/drive/MyDrive/Magistrale/Tesi/Fase2"

Mounted at /content/drive
/content/drive/MyDrive/Magistrale/Tesi/Fase2


In [9]:
SEEDS = [42, 8, 1291, 64207, 305, 91876, 12, 456, 33920, 7]

In [10]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
np.set_printoptions(threshold=sys.maxsize)

DATASET_INPUT_PATH = f"audio_dataset.csv"
#BEST_MODELS_CSV = f"best_models.csv"
#IMPORTANCES_DIR=Path("Importances")
#IMPORTANCES_DIR.mkdir(parents=True, exist_ok=True)
MODELS=["RandomForest","XGBoost","LightGBM","ElasticNet"]
TOP=1 #1
MAIN_METRIC='MAE' #'MAE'
START_FROM_SEED_INDEX=0
SUMMARY_CSV_PATH=Path(f"best_models_summary.csv")

In [11]:
MODELS_DIR = Path(f"ModelsToEvaluate")

MODELS_DIR.mkdir(parents=True, exist_ok=True)

CARICAMENTO DATI

In [12]:
df = pd.read_csv(DATASET_INPUT_PATH, index_col=0)

SELEZIONE MODELLI MIGLIORI

In [13]:
# 2. Calcoliamo il "Peso Hardware" per ogni combinazione testata
def train_evaluate_save_model(model_name, row, seed):

  # Estrazione parametri e metadati dalla riga
  parameters = ast.literal_eval(row['params'])
  rank=row["rank_test_score"]
  print(f"Allenando il modello di {model_name} n.{rank} (seed: {seed}) con parametri: {parameters} ...")
  current_df=df.copy()
  model={}

  # Inizializzazione del modello
  match model_name:
      case "RandomForest":
        model = RandomForestRegressor(**parameters, random_state=seed)
      case "XGBoost":
        if "device" in parameters: parameters.pop("device")
        model = xgb.XGBRegressor(**parameters, base_score = 0, tree_method='hist', random_state=seed)
      case "LightGBM":
        model = lgb.LGBMRegressor(**parameters, random_state=seed, verbosity=-1, device="cpu")
      case "ElasticNet":
        model = ElasticNet(**parameters, random_state=seed)
      case _:
        print("Modello non trovato")
        return

  # 2. PREPARAZIONE X e y
  target_column = 'portata'
  X = current_df.drop(columns=[target_column])
  y = current_df[target_column]

  # 3. SPLIT TRAIN/TEST
  # Dividiamo i dati: 80% training, 20% test
  #X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=seed)

  if model_name=="ElasticNet":
    print("Standardizzazione delle feature...")
    scaler = StandardScaler()
    scaler.set_output(transform="pandas")
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)
    pca = PCA(n_components=0.90, random_state=seed)
    pca.set_output(transform="pandas")

    print(f"Feature originali: {X_train.shape[1]}")

    # Trasforma i dati (X_train e X_test erano già stati scalati col tuo scaler)
    X_train = pca.fit_transform(X_train)
    X_test = pca.transform(X_test)

    print(f"Feature dopo PCA: {X_train.shape[1]}")

  # Addestramento del modello
  model.fit(X_train, y_train)

  # --- ESPORTAZIONE CON M2CGEN ---

  if model_name.lower() == "xgboost":
    # PATCH CRITICA PER XGBOOST:
    # m2cgen cerca l'attributo interno '_Booster' o 'booster'.
    # Nelle nuove versioni si chiama '_booster'. Creiamo un alias per m2cgen.
    if hasattr(model, "_booster"):
        model._Booster = model._booster
    elif hasattr(model, "get_booster"):
        model._Booster = model.get_booster()

  elif model_name.lower() == "lightgbm":
      # Per LightGBM usiamo il booster nativo compilato da .fit()
      if hasattr(model, "booster_"):
          model._booster = model.booster_

  filename_py=f"{model_name}_{rank}_{seed}.py"
  file_dimension_py=0
  file_dimension_mpy=0

  try:

    # Per XGBoost e LightGBM a volte serve passare il modello nativo, proviamo l'export standard
    python_code = m2c.export_to_python(model)

    # Definizione del percorso del file
    file_path = MODELS_DIR / filename_py

    with open(file_path, "w") as f:
        f.write(python_code)
    print(f"Modello salvato con successo in: {file_path}")

    # Crea il percorso completo del file .py di origine usando la variabile filename
    py_file = MODELS_DIR / filename_py

    try:
      file_dimension_py=py_file.stat().st_size
    except:
      pass

    # Genera il percorso del file .mpy finale cambiando l'estensione (.with_suffix)
    # e mantenendo la cartella di destinazione
    mpy_file = MODELS_DIR / Path(filename_py).with_suffix(".mpy")

    print(f"Compilazione in corso: {py_file.name} -> {mpy_file.name}")

    # Esegue mpy-cross convertendo i percorsi in stringhe (richiesto da subprocess)
    subprocess.run(
        ["mpy-cross", "-o", str(mpy_file), str(py_file)],
        check=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )

    try:
      file_dimension_mpy=mpy_file.stat().st_size
    except:
      pass

  except subprocess.CalledProcessError as e:
      print(f"[ERRORE] Impossibile compilare in MicroPython: {e.stderr.strip()}")
  except FileNotFoundError:
      print("[ERRORE CRITICO] Il comando 'mpy-cross' non è accessibile.")
  except Exception as e:
    print(f"Errore nell'esportazione MicroPython con m2cgen per {model_name}: {e}")

  filename_c=f"{model_name}_{rank}_{seed}.h"
  file_dimension_c=0

  try:

    # Per XGBoost e LightGBM a volte serve passare il modello nativo, proviamo l'export standard
    c_code = m2c.export_to_c(model)

    # Definizione del percorso del file
    file_path = MODELS_DIR / filename_c

    with open(file_path, "w") as f:
        f.write(c_code)
    print(f"Modello salvato con successo in: {file_path}")


    try:
      file_dimension_c=file_path.stat().st_size
    except:
      pass

  except Exception as e:
    print(f"Errore nell'esportazione C con m2cgen per {model_name}: {e}")



  # PREDIZIONE E VALUTAZIONE
  y_train_pred = model.predict(X_train)

  mae_training = mean_absolute_error(y_train, y_train_pred)
  me_training = np.mean(y_train - y_train_pred)
  mape_training = mean_absolute_percentage_error(y_train, y_train_pred) * 100
  r2_training = r2_score(y_train, y_train_pred)
  mse_training=mean_squared_error(y_train, y_train_pred)
  rmse_training = np.sqrt(mse_training)


  print(f"--- PERFORMANCE MODELLO (TRAINING) ---")
  print(f"R^2 Score: {r2_training:.4f}")
  print(f"MAE: {mae_training:.4f}")
  print(f"ME: {me_training:.4f}")
  print(f"MAPE: {mape_training:.2f}%")
  print(f"MSE: {mse_training:.4f}")
  print(f"RMSE: {rmse_training:.4f}")



  y_pred=model.predict(X_test)

  mae_test = mean_absolute_error(y_test, y_pred)
  me_test = np.mean(y_test - y_pred)
  mape_test = mean_absolute_percentage_error(y_test, y_pred) * 100
  r2_test = r2_score(y_test, y_pred)
  mse_test=mean_squared_error(y_test, y_pred)
  rmse_test = np.sqrt(mse_test)

  print(f"--- PERFORMANCE MODELLO (TESTING) ---")
  print(f"R^2 Score: {r2_test:.4f}")
  print(f"MAE: {mae_test:.4f}")
  print(f"ME: {me_test:.4f}")
  print(f"MAPE: {mape_test:.2f}%")
  print(f"MSE: {mse_test:.4f}")
  print(f"RMSE: {rmse_test:.4f}")

  info={
    "Modello": model_name,
    "Seed": str(seed),
    "Rank": str(rank),
    "Iperparametri": str(parameters),
    "MAE_Kfold": str(row["MAE"]),
    "R²_training": r2_training,
    "MAE_training": mae_training,
    "ME_training": me_training,
    "MAPE_training": f"{mape_training:.2f}%",
    "MSE_training": mse_training,
    "RMSE_training": rmse_training,
    "R²_test": r2_test,
    "MAE_test": mae_test,
    "ME_test": me_test,
    "MAPE_test": f"{mape_test:.2f}%",
    "MSE_test": mse_test,
    "RMSE_test": rmse_test,
    "Tempo di Inferenza": str(0),
    "Filename_py": filename_py,
    "Filename_mpy": filename_py.replace(".py",".mpy"),
    "Filename_c": filename_c,
    "DimensionePyhon": str(file_dimension_py),
    "DimensioneMicroPython": str(file_dimension_mpy),
    "DimensioneC": str(file_dimension_c)
  }

  df_info = pd.DataFrame([info])

  # 4. Salviamo in CSV usando i metodi nativi di Path
  # .exists() controlla se il file c'è già per decidere se fare l'append ('a') o la scrittura ('w')
  if SUMMARY_CSV_PATH.exists():
      df_info.to_csv(SUMMARY_CSV_PATH, mode='a', header=False, index=False)
  else:
      # Se il file non esiste, ci assicuriamo prima che la cartella di destinazione esista davvero
      df_info.to_csv(SUMMARY_CSV_PATH, mode='w', header=True, index=False)

  return

In [14]:
for i, seed in enumerate(SEEDS[START_FROM_SEED_INDEX:], start=START_FROM_SEED_INDEX):

  if START_FROM_SEED_INDEX != 0:
      print(f"Skipping the first {START_FROM_SEED_INDEX} seeds...")

  print(f"\n\n=== INIZIO RANDOMIZED SEARCH CON SEME {seed} ({i+1}/{len(SEEDS)}) ===\n")

  for model_name in MODELS:
    results_input_path=f"Results_randomized_search_{model_name.lower()}_{seed}.csv"
    # 1. Estraiamo i risultati della RandomizedSearch
    results = pd.read_csv(results_input_path)
    results[MAIN_METRIC] = -results['mean_test_score']
    top_mae = results.nsmallest(TOP, MAIN_METRIC).copy()
    top_mae.apply(
      lambda row: train_evaluate_save_model(model_name, row, seed),
      axis=1
    )



=== INIZIO RANDOMIZED SEARCH CON SEME 42 (1/10) ===

Allenando il modello di RandomForest n.1 (seed: 42) con parametri: {'n_estimators': 25, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_features': 0.8, 'max_depth': 10} ...
Modello salvato con successo in: ModelsToEvaluate/RandomForest_1_42.py
Compilazione in corso: RandomForest_1_42.py -> RandomForest_1_42.mpy
[ERRORE] Impossibile compilare in MicroPython: MemoryError:
Modello salvato con successo in: ModelsToEvaluate/RandomForest_1_42.h
--- PERFORMANCE MODELLO (TRAINING) ---
R^2 Score: 0.9899
MAE: 0.0119
ME: 0.0001
MAPE: 3.84%
MSE: 0.0003
RMSE: 0.0169
--- PERFORMANCE MODELLO (TESTING) ---
R^2 Score: 0.9691
MAE: 0.0184
ME: 0.0010
MAPE: 5.53%
MSE: 0.0008
RMSE: 0.0290
Allenando il modello di XGBoost n.1 (seed: 42) con parametri: {'subsample': 0.8, 'n_estimators': 25, 'max_depth': 9, 'learning_rate': 0.2, 'gamma': 0, 'colsample_bytree': 1.0} ...
Modello salvato con successo in: ModelsToEvaluate/XGBoost_1_42.py
Compilazione in cor

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.950e+01, tolerance: 2.146e-02 Linear regression models with null weight for the l1 regularization term are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeCV instead.
  model = cd_fast.enet_coordinate_descent(


Modello salvato con successo in: ModelsToEvaluate/ElasticNet_1_8.py
Compilazione in corso: ElasticNet_1_8.py -> ElasticNet_1_8.mpy
Modello salvato con successo in: ModelsToEvaluate/ElasticNet_1_8.h
--- PERFORMANCE MODELLO (TRAINING) ---
R^2 Score: 0.8452
MAE: 0.0446
ME: 0.0000
MAPE: 13.77%
MSE: 0.0043
RMSE: 0.0658
--- PERFORMANCE MODELLO (TESTING) ---
R^2 Score: 0.8255
MAE: 0.0476
ME: 0.0017
MAPE: 14.29%
MSE: 0.0048
RMSE: 0.0693


=== INIZIO RANDOMIZED SEARCH CON SEME 1291 (3/10) ===

Allenando il modello di RandomForest n.1 (seed: 1291) con parametri: {'n_estimators': 25, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_features': 0.8, 'max_depth': 10} ...
Modello salvato con successo in: ModelsToEvaluate/RandomForest_1_1291.py
Compilazione in corso: RandomForest_1_1291.py -> RandomForest_1_1291.mpy
[ERRORE] Impossibile compilare in MicroPython: MemoryError: memory allocation failed, allocating 144 bytes
Modello salvato con successo in: ModelsToEvaluate/RandomForest_1_1291.h
--- PE

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.999e+01, tolerance: 2.139e-02 Linear regression models with null weight for the l1 regularization term are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeCV instead.
  model = cd_fast.enet_coordinate_descent(


Modello salvato con successo in: ModelsToEvaluate/ElasticNet_1_1291.py
Compilazione in corso: ElasticNet_1_1291.py -> ElasticNet_1_1291.mpy
Modello salvato con successo in: ModelsToEvaluate/ElasticNet_1_1291.h
--- PERFORMANCE MODELLO (TRAINING) ---
R^2 Score: 0.8414
MAE: 0.0455
ME: -0.0000
MAPE: 13.99%
MSE: 0.0044
RMSE: 0.0665
--- PERFORMANCE MODELLO (TESTING) ---
R^2 Score: 0.8483
MAE: 0.0450
ME: -0.0020
MAPE: 14.25%
MSE: 0.0042
RMSE: 0.0649


=== INIZIO RANDOMIZED SEARCH CON SEME 64207 (4/10) ===

Allenando il modello di RandomForest n.1 (seed: 64207) con parametri: {'n_estimators': 25, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_features': 0.8, 'max_depth': 10} ...
Modello salvato con successo in: ModelsToEvaluate/RandomForest_1_64207.py
Compilazione in corso: RandomForest_1_64207.py -> RandomForest_1_64207.mpy
[ERRORE] Impossibile compilare in MicroPython: MemoryError:
Modello salvato con successo in: ModelsToEvaluate/RandomForest_1_64207.h
--- PERFORMANCE MODELLO (TRAINING

...

Modello salvato con successo in: ModelsToEvaluate/ElasticNet_1_456.py
Compilazione in corso: ElasticNet_1_456.py -> ElasticNet_1_456.mpy
Modello salvato con successo in: ModelsToEvaluate/ElasticNet_1_456.h
--- PERFORMANCE MODELLO (TRAINING) ---
R^2 Score: 0.8411
MAE: 0.0455
ME: 0.0000
MAPE: 13.99%
MSE: 0.0044
RMSE: 0.0667
--- PERFORMANCE MODELLO (TESTING) ---
R^2 Score: 0.8427
MAE: 0.0448
ME: -0.0013
MAPE: 13.88%
MSE: 0.0043
RMSE: 0.0658


=== INIZIO RANDOMIZED SEARCH CON SEME 33920 (9/10) ===

Allenando il modello di RandomForest n.1 (seed: 33920) con parametri: {'n_estimators': 25, 'min_samples_split': 15, 'min_samples_leaf': 4, 'max_features': 0.5, 'max_depth': 10} ...
Modello salvato con successo in: ModelsToEvaluate/RandomForest_1_33920.py
Compilazione in corso: RandomForest_1_33920.py -> RandomForest_1_33920.mpy
[ERRORE] Impossibile compilare in MicroPython: MemoryError: memory allocation failed, allocating 131072 bytes
Modello salvato con successo in: ModelsToEvaluate/RandomFore

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.911e+01, tolerance: 2.142e-02 Linear regression models with null weight for the l1 regularization term are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeCV instead.
  model = cd_fast.enet_coordinate_descent(


Modello salvato con successo in: ModelsToEvaluate/ElasticNet_1_33920.py
Compilazione in corso: ElasticNet_1_33920.py -> ElasticNet_1_33920.mpy
Modello salvato con successo in: ModelsToEvaluate/ElasticNet_1_33920.h
--- PERFORMANCE MODELLO (TRAINING) ---
R^2 Score: 0.8482
MAE: 0.0442
ME: -0.0000
MAPE: 13.64%
MSE: 0.0042
RMSE: 0.0651
--- PERFORMANCE MODELLO (TESTING) ---
R^2 Score: 0.8101
MAE: 0.0481
ME: 0.0038
MAPE: 14.37%
MSE: 0.0053
RMSE: 0.0725


=== INIZIO RANDOMIZED SEARCH CON SEME 7 (10/10) ===

Allenando il modello di RandomForest n.1 (seed: 7) con parametri: {'n_estimators': 25, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_features': 0.8, 'max_depth': 10} ...
Modello salvato con successo in: ModelsToEvaluate/RandomForest_1_7.py
Compilazione in corso: RandomForest_1_7.py -> RandomForest_1_7.mpy
[ERRORE] Impossibile compilare in MicroPython: MemoryError:
Modello salvato con successo in: ModelsToEvaluate/RandomForest_1_7.h
--- PERFORMANCE MODELLO (TRAINING) ---
R^2 Score: 0.9

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.962e+01, tolerance: 2.127e-02 Linear regression models with null weight for the l1 regularization term are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeCV instead.
  model = cd_fast.enet_coordinate_descent(


Modello salvato con successo in: ModelsToEvaluate/ElasticNet_1_7.py
Compilazione in corso: ElasticNet_1_7.py -> ElasticNet_1_7.mpy
Modello salvato con successo in: ModelsToEvaluate/ElasticNet_1_7.h
--- PERFORMANCE MODELLO (TRAINING) ---
R^2 Score: 0.8432
MAE: 0.0450
ME: -0.0000
MAPE: 13.88%
MSE: 0.0043
RMSE: 0.0659
--- PERFORMANCE MODELLO (TESTING) ---
R^2 Score: 0.8351
MAE: 0.0460
ME: 0.0015
MAPE: 14.00%
MSE: 0.0047
RMSE: 0.0685


In [15]:
  # 1. Carica il file
  df = pd.read_csv(SUMMARY_CSV_PATH)

  # 3. Ordina in base al MAE (dal più piccolo al più grande)
  df_ordered = df.sort_values(by='MAE_test', ascending=True)

  # 4. Genera il nome del file di output
  file_base, file_ext = os.path.splitext(SUMMARY_CSV_PATH)
  output_file = f"{file_base}_ordered{file_ext}"

  # 5. Salva il file
  df_ordered.to_csv(output_file, index=False)

  print(f"File salvato: {output_file}")

File salvato: best_models_summary_ordered.csv
